In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

class HouseKgParser:
    def __init__(self, max_workers=10):
        self.base_url = "https://www.house.kg"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        self.max_workers = max_workers

    def get_soup(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            if response.status_code == 200:
                return BeautifulSoup(response.text, 'html.parser')
        except Exception as e:
            print(f"Ошибка запроса {url}: {e}")
        return None

    def get_listing_links(self, page):
        url = f"{self.base_url}/kupit-kvartiru?page={page}"
        soup = self.get_soup(url)
        if not soup: return []

        links = []
        for a in soup.find_all('a', href=re.compile(r'/details/')):
            href = a.get('href')
            if href:
                full_url = self.base_url + href if href.startswith('/') else href
                links.append(full_url)
        return list(dict.fromkeys(links))

    def extract_coordinates(self, soup):
        # Извлекает координаты напрямую из атрибутов div-карты
        map_div = soup.find('div', id='map2gis')
        if map_div:
            lat = map_div.get('data-lat')
            lon = map_div.get('data-lon')
            try:
                return float(lat), float(lon)
            except (TypeError, ValueError):
                pass
        return None, None

    def parse_detail_page(self, url):
        soup = self.get_soup(url)
        if not soup: return None

        data = {}

        try:
            # Заголовок -> main
            h1 = soup.find('h1')
            data['main'] = h1.get_text(strip=True) if h1 else None

            # Адрес -> address
            addr_div = soup.find('div', class_='address')
            data['address'] = addr_div.get_text(strip=True) if addr_div else None

            # Добавлено -> added
            added_span = soup.find('span', class_='added-span')
            data['added'] = added_span.get_text(strip=True) if added_span else None

            # Поднято -> upped
            upped_span = soup.find('span', class_='upped-span')
            data['upped'] = upped_span.get_text(strip=True) if upped_span else None

            # Просмотры -> view_count
            view_span = soup.find('span', class_='view-count')
            if view_span:
                view_text = view_span.get_text(strip=True)
                match = re.search(r'\d+', view_text)
                data['view_count'] = int(match.group()) if match else None
            else:
                data['view_count'] = None

            # Лайки -> hearts
            hearts_span = soup.find('span', class_='favourite-count')
            if hearts_span:
                hearts_text = hearts_span.get_text(strip=True)
                match = re.search(r'\d+', hearts_text)
                data['hearts'] = int(match.group()) if match else None
            else:
                data['hearts'] = None

            # Координаты -> lat, lon
            lat, lon = self.extract_coordinates(soup)
            data['lat'] = lat
            data['lon'] = lon

            # Цена в долларах -> usd_price
            p_usd = soup.find('div', class_='price-dollar')
            if p_usd:
                price_text = re.sub(r'[^\d]', '', p_usd.get_text(strip=True))
                data['usd_price'] = int(price_text) if price_text else None
            else:
                data['usd_price'] = None

            # Описание от продавца
            desc_div = soup.find('div', class_='description')
            if desc_div:
                desc_p = desc_div.find('p', class_='comment')
                data['Описание от продавца'] = desc_p.get_text(strip=True) if desc_p else None
            else:
                data['Описание от продавца'] = None

            # Характеристики из info-row
            info_rows = soup.find_all('div', class_='info-row')
            for row in info_rows:
                label = row.find('div', class_='label')
                info = row.find('div', class_='info')
                if label and info:
                    key = label.get_text(strip=True)
                    data[key] = info.get_text(strip=True)

        except Exception as e:
            print(f"Ошибка парсинга {url}: {e}")

        return data

    def run(self, start_page=1, end_page=2):
        print(f"🚀 Сбор ссылок (стр. {start_page}-{end_page})...")
        all_links = []
        for p in range(start_page, end_page + 1):
            links = self.get_listing_links(p)
            all_links.extend(links)

        all_links = list(dict.fromkeys(all_links))
        print(f"🔗 Найдено объявлений: {len(all_links)}")

        results = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_url = {executor.submit(self.parse_detail_page, url): url for url in all_links}
            for future in tqdm(as_completed(future_to_url), total=len(all_links), desc="Парсинг"):
                res = future.result()
                if res: results.append(res)

        df = pd.DataFrame(results)

        # Приводим к структуре train (1).csv
        train_cols = [
            'main', 'address', 'added', 'upped', 'view_count', 'hearts', 'lat', 'lon',
            'Тип предложения', 'Серия', 'Дом', 'Этаж', 'Площадь', 'Отопление',
            'Состояние', 'Газ', 'Санузел', 'Балкон', 'Входная дверь', 'Парковка',
            'Высота потолков', 'Безопасность', 'Правоустанавливающие документы',
            'Телефон', 'Интернет', 'Мебель', 'Возможность рассрочки', 'Возможность обмена',
            'Разное', 'Пол', 'Возможность ипотеки', 'Площадь участка', 'Канализация',
            'Питьевая вода', 'Электричество', 'usd_price', 'Описание от продавца'
        ]

        for col in train_cols:
            if col not in df.columns:
                df[col] = None

        df = df[train_cols]

        filename = f"house_kg_sell_flats_{int(time.time())}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✅ Готово! Файл: {filename}")
        return df

if __name__ == '__main__':
    parser = HouseKgParser(max_workers=15)
    df = parser.run(start_page=1, end_page=1256)


🚀 Сбор ссылок (стр. 1-1256)...
🔗 Найдено объявлений: 12004


Парсинг:  59%|█████▉    | 7097/12004 [16:30<45:03,  1.81it/s]  

Ошибка запроса https://www.house.kg/details/9802476a468617670599-54591972: HTTPSConnectionPool(host='www.house.kg', port=443): Read timed out. (read timeout=15)


Парсинг:  59%|█████▉    | 7099/12004 [16:30<34:56,  2.34it/s]

Ошибка запроса https://www.house.kg/details/700233069f0574ec458f1-78834881: HTTPSConnectionPool(host='www.house.kg', port=443): Read timed out. (read timeout=15)
Ошибка запроса https://www.house.kg/details/478836369dcaa1b2eb361-41552454: HTTPSConnectionPool(host='www.house.kg', port=443): Read timed out. (read timeout=15)
Ошибка запроса https://www.house.kg/details/76656616a2f887106cf99-01928305: HTTPSConnectionPool(host='www.house.kg', port=443): Read timed out. (read timeout=15)
Ошибка запроса https://www.house.kg/details/188948269e0abf12f3104-45529320: HTTPSConnectionPool(host='www.house.kg', port=443): Max retries exceeded with url: /details/188948269e0abf12f3104-45529320 (Caused by ConnectTimeoutError(<HTTPSConnection(host='www.house.kg', port=443) at 0x21d12193800>, 'Connection to www.house.kg timed out. (connect timeout=15)'))


Парсинг:  59%|█████▉    | 7108/12004 [16:31<10:12,  8.00it/s]

Ошибка запроса https://www.house.kg/details/145955969d8dd3d6fed71-00923047: HTTPSConnectionPool(host='www.house.kg', port=443): Read timed out. (read timeout=15)


Парсинг: 100%|██████████| 12004/12004 [28:00<00:00,  7.14it/s]


✅ Готово! Файл: house_kg_sell_flats_1783230474.csv
